<a href="https://colab.research.google.com/github/n1m1nd33p/Practical-ML/blob/main/Loan_Approval_Prediction_ML_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Loan Approval Prediction Using Machine Learning
**B.Tech AIML Mini Project | Google Colab**



**Dataset:** Kaggle – [Loan Prediction Problem Dataset](https://www.kaggle.com/datasets/altruistdelhite04/loan-prediction-problem-dataset)  
Use the training CSV (usually named `train_u6lujuX_CVtuZ9i.csv` or `train.csv`).




## 1. Import required libraries
We use pandas and NumPy for data handling, Matplotlib/Seaborn for visualization, and scikit-learn for preprocessing, model training and evaluation.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

import warnings
warnings.filterwarnings('ignore')
print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Upload the Kaggle dataset
Download the Kaggle dataset, extract the ZIP file, and upload its **training CSV** below. Do not upload the test CSV because it does not contain the target column.

In [ ]:
uploaded = files.upload()
file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
display(df.head())

## 3. Understand the dataset
`Loan_Status` is the target column: `Y` means approved and `N` means rejected. The remaining useful columns are input features. `Loan_ID` is only an identifier, so we remove it before training.

In [ ]:
print("Dataset information:")
df.info()

print("\\nMissing values in each column:")
display(df.isnull().sum())

print("\\nTarget distribution:")
display(df['Loan_Status'].value_counts())

## 4. Exploratory Data Analysis (EDA)
We inspect the target distribution and a few numerical columns to understand the data before training.

In [ ]:
plt.figure(figsize=(5,4))
sns.countplot(data=df, x='Loan_Status')
plt.title('Loan Approval Status Distribution')
plt.xlabel('Loan Status (Y = Approved, N = Rejected)')
plt.show()

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
df[numeric_cols].hist(figsize=(12,8), bins=20)
plt.suptitle('Distribution of Numerical Features')
plt.tight_layout()
plt.show()

## 5. Separate input features and target
We remove the identifier and separate `X` (input features) from `y` (the answer the model learns to predict).

In [ ]:
# Remove rows where the target is missing, if any
df = df.dropna(subset=['Loan_Status']).copy()

X = df.drop(columns=['Loan_Status', 'Loan_ID'], errors='ignore')
y = df['Loan_Status'].astype(str).str.strip().str.upper()

# Convert Y/N labels to 1/0
y = y.map({'Y': 1, 'N': 0})
if y.isnull().any():
    raise ValueError("Unexpected target labels. Check the Loan_Status column.")

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("1 = Approved, 0 = Rejected")

## 6. Preprocess the data
The dataset contains both numerical and categorical columns and may contain missing values.

- Numerical missing values are filled with the median.
- Categorical missing values are filled with the most frequent value.
- Categorical values are converted into numeric columns using One-Hot Encoding.
- Numerical values are standardized for Logistic Regression.

Using a scikit-learn Pipeline ensures preprocessing is learned only from the training data and helps prevent data leakage.

In [ ]:
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

## 7. Split data into training and testing sets
We use 80% of the data to train the models and 20% to test them. Stratification keeps the approval/rejection proportions similar in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])

## 8. Train three Machine Learning models
1. **Logistic Regression:** A classification algorithm that estimates the probability of a class.
2. **Decision Tree:** Makes predictions using a sequence of feature-based decisions.
3. **Random Forest:** Combines predictions from many decision trees.

All three models use the same preprocessing steps for a fair comparison.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=4),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, max_depth=6)
}

trained_models = {}
results = []

for name, estimator in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', estimator)
    ])
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    trained_models[name] = pipeline
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, predictions),
        'Precision': precision_score(y_test, predictions, zero_division=0),
        'Recall': recall_score(y_test, predictions, zero_division=0),
        'F1 Score': f1_score(y_test, predictions, zero_division=0)
    })

results_df = pd.DataFrame(results).sort_values(by='F1 Score', ascending=False)
display(results_df.round(3))

## 9. Compare model performance
Accuracy is the percentage of all test predictions that are correct. Precision, recall and F1-score provide additional information, which is useful when the classes are not equally represented.

In [ ]:
plt.figure(figsize=(8,4))
sns.barplot(data=results_df, x='Model', y='Accuracy')
plt.ylim(0, 1)
plt.title('Accuracy Comparison of ML Models')
plt.ylabel('Accuracy')
plt.xticks(rotation=15)
plt.show()

## 10. Evaluate the best model
We select the model with the highest F1-score on the test set for this demonstration. The confusion matrix shows correct and incorrect predictions.

In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_model = trained_models[best_model_name]
best_predictions = best_model.predict(X_test)

print("Best model by test F1-score:", best_model_name)
print("\\nClassification Report:")
print(classification_report(y_test, best_predictions,
                            target_names=['Rejected (0)', 'Approved (1)'],
                            zero_division=0))

cm = confusion_matrix(y_test, best_predictions)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Rejected', 'Approved'],
            yticklabels=['Rejected', 'Approved'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

## 11. Test the model on a new applicant
The example below uses the same feature columns as the uploaded dataset. Values are illustrative. Change them to experiment with different applicant profiles.

In [ ]:
# Create one sample applicant using the dataset's feature columns.
sample = X.iloc[[0]].copy()

# Change selected values only when those columns exist in this dataset.
example_values = {
    'Gender': 'Male',
    'Married': 'Yes',
    'Dependents': '1',
    'Education': 'Graduate',
    'Self_Employed': 'No',
    'ApplicantIncome': 6000,
    'CoapplicantIncome': 2000,
    'LoanAmount': 150,
    'Loan_Amount_Term': 360,
    'Credit_History': 1,
    'Property_Area': 'Urban'
}

for column, value in example_values.items():
    if column in sample.columns:
        sample.loc[sample.index[0], column] = value

prediction = best_model.predict(sample)[0]
print("Predicted result:", "LOAN APPROVED" if prediction == 1 else "LOAN REJECTED")

if hasattr(best_model.named_steps['model'], 'predict_proba'):
    probability = best_model.predict_proba(sample)[0][1]
    print(f"Model-estimated approval probability: {probability:.2%}")